# 03. API 수집 → 파이프라인 연결
> 공공데이터 API로 실시간 수집한 데이터를 pipeline.py에 바로 연결한다.

| 단계 | 내용 |
|------|------|
| STEP 1 | 환경 세팅 & API 키 로드 |
| STEP 2 | collector로 데이터 수집 |
| STEP 3 | normalize_columns로 컬럼 변환 |
| STEP 4 | pipeline에 입력 & 수요 점수 산출 |
| STEP 5 | 결과 저장 |

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.collector import ApartmentDataCollector, SEOUL_SIGUNGU_CODES
from src.pipeline import DemandForecastingPipeline

print('라이브러리 로드 완료 ✅')

라이브러리 로드 완료 ✅


In [2]:
import os

API_KEY = input("API 키를 입력하세요: ")

print("API 키 설정 완료 ✅")

API 키 설정 완료 ✅


In [3]:
collector = ApartmentDataCollector(api_key=API_KEY)

# 테스트
df_test = collector.fetch_one(sigungu_code="11650", year_month="202602")

print(f"수집 건수: {len(df_test):,}건")
df_test.head(3)

수집 건수: 98건


,aptDong,aptNm,aptSeq,bonbun,bubun,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,...,roadNmCd,roadNmSeq,roadNmSggCd,roadNmbCd,sggCd,slerGbn,umdCd,umdNm,수집_시군구코드,수집_연월
0,,상지리츠빌4차(843-13),11650-646,0843,0013,2004,개인,,,"225,000",...,4163401,01,11650,0,11650,개인,10100,방배동,11650,202602
1,,대우아이빌,11650-101,0042,0013,2001,개인,,,"66,500",...,4163459,01,11650,0,11650,개인,10600,잠원동,11650,202602
2,,래미안서초유니빌,11650-518,1445,0004,2003,개인,,,"48,500",...,3121024,03,11650,0,11650,개인,10800,서초동,11650,202602


In [4]:
# 서울 25개 구 최근 12개월 수집
df_raw = collector.fetch_recent_months(
    months=12,
    save_path="../data/raw_api_collected.csv"
)

print(f"\n수집 완료: {len(df_raw):,}건 ✅")

[AUTO] 최근 12개월: 202504 ~ 202603
[COLLECT] 25개 구 × 12개월 = 총 300회 호출 시작
          202504 ~ 202603

  [  1/300] 종로구(11110) / 202504 ... 43건 ✓
  [  2/300] 종로구(11110) / 202505 ... 61건 ✓
  [  3/300] 종로구(11110) / 202506 ... 107건 ✓
  [  4/300] 종로구(11110) / 202507 ... 40건 ✓
  [  5/300] 종로구(11110) / 202508 ... 42건 ✓
  [  6/300] 종로구(11110) / 202509 ... 67건 ✓
  [  7/300] 종로구(11110) / 202510 ... 88건 ✓
  [  8/300] 종로구(11110) / 202511 ... 22건 ✓
  [  9/300] 종로구(11110) / 202512 ... 43건 ✓
  [ 10/300] 종로구(11110) / 202601 ... 46건 ✓
  [ 11/300] 종로구(11110) / 202602 ... 44건 ✓
  [ 12/300] 종로구(11110) / 202603 ... 7건 ✓
  [ 13/300] 중구(11140) / 202504 ... 109건 ✓
  [ 14/300] 중구(11140) / 202505 ... 127건 ✓
  [ 15/300] 중구(11140) / 202506 ... 216건 ✓
  [ 16/300] 중구(11140) / 202507 ... 56건 ✓
  [ 17/300] 중구(11140) / 202508 ... 71건 ✓
  [ 18/300] 중구(11140) / 202509 ... 162건 ✓
  [ 19/300] 중구(11140) / 202510 ... 146건 ✓
  [ 20/300] 중구(11140) / 202511 ... 34건 ✓
  [ 21/300] 중구(11140) / 202512 ... 78건 ✓
  [ 22/300] 중구(11140) / 2

In [5]:
# API 영문 컬럼 → pipeline 한글 컬럼 변환
df_normalized = collector.normalize_columns(df_raw)

print(f"변환 후 컬럼: {df_normalized.columns.tolist()}")
df_normalized.head(3)

변환 후 컬럼: ['시군구', '단지명', '전용면적(㎡)', '계약년월', '거래금액(만원)', '건축년도', '도로명', '수집_시군구코드', '수집_연월']


,시군구,단지명,전용면적(㎡),계약년월,거래금액(만원),건축년도,도로명,수집_시군구코드,수집_연월
0,서울특별시 종로구 숭인동,롯데캐슬천지인,84.95,202504,"98,000",2004,종로,11110,202504
1,서울특별시 종로구 신영동,대아파크빌,84.84,202504,"65,500",1998,진흥로,11110,202504
2,서울특별시 종로구 창신동,창신쌍용2,115.53,202504,"79,000",1993,낙산길,11110,202504


In [6]:
# API로 수집한 데이터를 pipeline에 직접 입력
pipeline = DemandForecastingPipeline(
    supply_path="../data/한국부동산원_주택공급정보_입주예정물량정보_20251231.csv"
)

df_result, sido_summary = pipeline.run(df_transactions=df_normalized)

print("\n=== TOP 5 ===")
df_result.head()

  오늘의집 인테리어 수요 예측 파이프라인 시작
[LOAD] 외부 DataFrame 입력: 73,636건
[LOAD] 입주예정 로드: 675건
[PREPROCESS] 실거래가 정제 완료: 73,636건
  세그먼트 분포:
세그먼트
Very_Old_Apartment    43185
Mid_Apartment         12644
Old_Apartment         10017
New_Apartment          7790
[PREPROCESS] 입주예정 정제 완료: 348건
[MERGE] 집계 완료: 25개 시군구
[SCORE] 수요 점수 계산 완료
  최고: 70.7 | 평균: 34.7

=== TOP 5 ===


,시도,시군구,거래건수,평균거래금액_만원,평균노후도_년,평균면적_m2,신규입주_세대수,입주단지수,인테리어_수요점수
14,서울,서초구,382,289697.1,18.1,109.6,5886.0,7.0,70.73
17,서울,송파구,944,248433.8,17.7,86.7,2572.0,4.0,59.90
0,서울,강남구,416,282072.6,18.4,90.1,349.0,2.0,55.19
16,서울,성북구,1503,90274.6,17.9,79.2,856.0,3.0,49.35
5,서울,광진구,247,180512.2,18.2,108.5,828.0,3.0,47.78


In [7]:
print("=== 시도별 수요 집계 ===")
sido_summary

=== 시도별 수요 집계 ===


,시도,시군구수,총거래건수,평균수요점수,최고수요점수,총신규입주
0,서울,25,10017,34.71,70.73,27158.0


In [8]:
# 결과 CSV 저장
df_result.to_csv(
    "../data/인테리어_수요점수_결과.csv",
    index=False,
    encoding="utf-8-sig"
)

sido_summary.to_csv(
    "../data/시도별_수요집계_요약.csv",
    index=False,
    encoding="utf-8-sig"
)

print("결과 저장 완료 ✅")
print("  • data/인테리어_수요점수_결과.csv")
print("  • data/시도별_수요집계_요약.csv")

결과 저장 완료 ✅
  • data/인테리어_수요점수_결과.csv
  • data/시도별_수요집계_요약.csv
